# 🔄 Spaced Repetition & Integrasi Pipeline Akhir
---
Notebook ini merupakan integrasi akhir **(Fase 4 & 6)**. Kita akan:
1. Mengimplementasikan algoritma **Spaced Repetition (FSRS / SM-2)** untuk penjadwalan review mandiri.
2. Memadukan **Knowledge Graph (NetworkX)**, **Knowledge Tracing (pyBKT)**, dan **Spaced Repetition** menjadi satu kesatuan mesin rekomendasi.
3. Membuat **Simulasi Chatbot Interaktif berbasis CLI** di bagian akhir agar Anda bisa mencoba berinteraksi langsung sebelum memprogram Frontend Next.js.

### 1. Setup & Instalasi Dependensi

In [ ]:
# Install library fsrs dan pyBKT
!pip install fsrs pyBKT

import pandas as pd
import numpy as np
import networkx as nx
import os
from pyBKT.models import Model
from fsrs import Scheduler, Card, Rating
from datetime import datetime, timedelta

# Menghubungkan ke Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/AI-Learning Path/'
except:
    print("Berjalan secara lokal.")
    BASE_PATH = './'

print("✅ Setup selesai!")

### 2. Memuat Seluruh Dataset OULAD

In [ ]:
%%time
print("Memuat data OULAD...")
courses = pd.read_csv(os.path.join(BASE_PATH, 'courses.csv'))
assessments = pd.read_csv(os.path.join(BASE_PATH, 'assessments.csv'))
vle = pd.read_csv(os.path.join(BASE_PATH, 'vle.csv'))
studentInfo = pd.read_csv(os.path.join(BASE_PATH, 'studentInfo.csv'))
studentAssessment = pd.read_csv(os.path.join(BASE_PATH, 'studentAssessment.csv'))
studentVle = pd.read_csv(os.path.join(BASE_PATH, 'studentVle.csv'), nrows=1000000) # Batasi 1M untuk memori Colab

print("✅ Semua file OULAD berhasil dimuat!")

### 3. Komponen Pendukung Pipeline (Graph & BKT)
Kita buat pembungkus agar pipeline berjalan otomatis.

In [ ]:
class AdaptiveLearningEngine:
    def __init__(self, target_module='AAA'):
        self.target_module = target_module
        self.G = nx.DiGraph()
        self.bkt_model = Model(seed=42, num_fits=1)
        self.scheduler = Scheduler()
        self.student_cards = {} # Menyimpan card FSRS per siswa dan konsep
        
    def build_kg(self, vle_df, assessments_df):
        # Bangun node graf konsep
        vle_sub = vle_df[vle_df['code_module'] == self.target_module]
        asst_sub = assessments_df[assessments_df['code_module'] == self.target_module]
        
        for _, r in vle_sub.iterrows():
            self.G.add_node(f"Materi_{r['id_site']}", type=r['activity_type'], label=r['activity_type'].upper())
            
        for _, r in asst_sub.iterrows():
            self.G.add_node(f"Tugas_{r['id_assessment']}", type=r['assessment_type'], label=f"KUIS {r['assessment_type']}")
            
        # Hubungkan linear sebagai baseline prerequisite
        nodes = list(self.G.nodes())
        for i in range(len(nodes)-1):
            self.G.add_edge(nodes[i], nodes[i+1])
            
        print(f"Graf berhasil dibangun: {len(self.G.nodes())} Node konsep")

    def train_bkt(self, sa_df, a_df):
        # Format data BKT
        df_m = sa_df.merge(a_df, on='id_assessment')
        df_m['correct'] = (df_m['score'] >= 40).astype(int)
        df_m = df_m.sort_values(['id_student', 'date_submitted'])
        df_m['order_id'] = df_m.groupby('id_student').cumcount() + 1
        
        bkt_data = df_m.rename(columns={'id_student': 'user_id', 'assessment_type': 'skill_name'})
        bkt_data = bkt_data[['user_id', 'skill_name', 'correct', 'order_id']]
        
        print("Melatih model BKT untuk memprediksi tingkat pemahaman...")
        self.bkt_model.fit(data=bkt_data)
        print("Model BKT siap digunakan!")
        
engine = AdaptiveLearningEngine(target_module='AAA')
engine.build_kg(vle, assessments)
engine.train_bkt(studentAssessment, assessments)

### 4. Sistem Spaced Repetition (FSRS)
Menghitung kapan materi yang sudah dikuasai harus di-review agar tidak lupa (Forgetting Curve).

In [ ]:
class SpacedRepetitionManager:
    def __init__(self, scheduler):
        self.scheduler = scheduler
        self.cards = {} # concept_id -> Card()
        
    def review_concept(self, concept_id, rating_choice):
        if concept_id not in self.cards:
            self.cards[concept_id] = Card()
            
        card = self.cards[concept_id]
        
        # Konversi rating ke Enum FSRS
        ratings = {
            1: Rating.Again,  # Lupa total
            2: Rating.Hard,   # Susah ingat
            3: Rating.Good,   # Ingat dengan baik
            4: Rating.Easy    # Sangat gampang
        }
        
        selected_rating = ratings.get(rating_choice, Rating.Good)
        # Update kartu kognitif
        new_card, review_log = self.scheduler.review_card(card, selected_rating, datetime.now())
        self.cards[concept_id] = new_card
        return new_card

### 5. 🤖 Uji Coba Chatbot Interaktif (CLI Prototype)
Jalankan sel di bawah ini untuk memulai **interaksi simulasi dengan AI Learning Path Chatbot**. 

Anda bisa menyamar sebagai mahasiswa, meminta rekomendasi belajar berdasarkan Knowledge Graph, menyelesaikan tugas (mengubah skor BKT), dan menjadwalkan review ulang materi menggunakan FSRS secara langsung di dalam Google Colab!

In [ ]:
def start_chatbot_sim(engine):
    import re
    from datetime import datetime
    
    print("\n=======================================================")
    print("🛡️ AI Learning Path Chatbot Simulator - ETHICAL GUARDRAILS VERSION 🛡️")
    print("=======================================================")
    
    # Mock Student Profiles (showing Bias Mitigation)
    student_profiles = {
        1: {"id": 240030, "name": "Tubagus", "ses": "low", "disability": "Y", "label": "Low SES, Disabled (Marginalized)"},
        2: {"id": 999999, "name": "Budi", "ses": "high", "disability": "N", "label": "High SES, Non-disabled (Privileged)"}
    }
    
    print("Pilih profil mahasiswa untuk simulasi:")
    for k, v in student_profiles.items():
        print(f"  {k}. {v['name']} ({v['label']})")
    
    prof_choice = input("Pilih profil (1-2, default 1): ")
    prof_choice = int(prof_choice) if prof_choice in ['1', '2'] else 1
    active_student = student_profiles[prof_choice]
    
    print(f"\n✅ Mahasiswa Aktif: {active_student['name']} (ID: {active_student['id']})")
    print(f"   Demografi: SES={active_student['ses'].upper()} | Disability={active_student['disability']}")
    print("   [MITIGASI BIAS #1]: Sistem menggunakan parameter BKT yang 100% KOGNITIF-MURNI.")
    print("   Status sosial-ekonomi dan disabilitas tidak memengaruhi perhitungan probabilitas kelulusan.\n")
    
    sr_manager = SpacedRepetitionManager(engine.scheduler)
    
    # Mock Mastery awal
    nodes = list(engine.G.nodes())
    mastery_scores = {node: 0.0 for node in nodes}
    
    # Set data awal
    mastery_scores[nodes[0]] = 0.95
    mastery_scores[nodes[1]] = 0.85
    
    # Rem-loop counter to demonstrate Challenge Injection
    remedial_attempts = 0
    
    while True:
        print("\n--- MENU CHATBOT (ETHICAL GUARDRAILS INTEGRATED) ---")
        print("1. 📊 Progress & Path (Transparansi / Explainable AI)")
        print("2. 💡 Rekomendasi Materi Selanjutnya (Mitigasi Bias & Challenge Injection)")
        print("3. 💬 Chat dengan AI Tutor (Filter Plagiarisme / Socratic Guard)")
        print("4. 📝 Laporkan Penyelesaian / Nilai Kuis (Update BKT & Spaced Repetition)")
        print("5. 🛡️ Report / Override Manusia (Akuntabilitas / Human-in-the-Loop)")
        print("6. 🔒 Data Privacy Log (Security Threats & Data Minimization)")
        print("7. 🚪 Keluar")
        
        pilihan = input("\nPilih menu (1-7): ")
        
        if pilihan == '1':
            # TRANSPARANSI (Explainable AI)
            print(f"\n=======================================================")
            print(f"📊 PROGRES BELAJAR: {active_student['name']}")
            print(f"=======================================================")
            mastered = [n for n, s in mastery_scores.items() if s >= 0.8]
            unmastered = [n for n, s in mastery_scores.items() if s < 0.8]
            print(f"• Total Konsep dikuasai: {len(mastered)} / {len(nodes)} ({len(mastered)/len(nodes):.1%})")
            print(f"• Konsep Belum Dikuasai: {len(unmastered)}")
            
            # Tampilkan penjelasan transparan kenapa konsep direkomendasikan
            print("\n📋 Peta Transparansi Keputusan (Explainable AI):")
            for node in nodes[:5]:
                status = "✅ LULUS" if mastery_scores[node] >= 0.8 else "❌ BELUM LULUS"
                # Cari predecessor
                preds = list(engine.G.predecessors(node))
                pred_status = "Semua prasyarat terpenuhi." if all(mastery_scores.get(p, 0) >= 0.8 for p in preds) else "Ada prasyarat belum tuntas."
                print(f"  - {node}: {status} (Skor: {mastery_scores[node]:.0%})")
                print(f"    AI Explanation: {pred_status} Direkomendasikan berdasarkan graph linear.")
                
        elif pilihan == '2':
            # MITIGASI BIAS & CHALLENGE INJECTION
            # Cari materi pertama yang belum dikuasai
            next_node = None
            for node in nodes:
                if mastery_scores[node] < 0.8:
                    next_node = node
                    break
            
            if next_node:
                print(f"\n💡 [AI RECOMMENDATION] - TRANSPARAN & ADAPTIF:")
                print(f"   👉 Materi selanjutnya: {next_node} (Tipe: {engine.G.nodes[next_node].get('type')})")
                
                # Cek jika mahasiswa berada dalam Remedial Loop
                if remedial_attempts >= 2:
                    print("   [CHALLENGE INJECTION ACTIVATED! 🚀]")
                    print("   AI mendeteksi mahasiswa mengulang materi ini berkali-kali (Potensi Remedial Loop).")
                    print("   Untuk mencegah bias frustrasi dan diskriminasi sistemik, AI menginjeksikan:")
                    print("   1. Challenge alternatif visual/interaktif.")
                    print("   2. Link bimbingan 1-on-1 dengan Asisten Praktikum.")
                    print(f"   3. Akses sementara ke konsep tingkat lanjut '{nodes[min(len(nodes)-1, nodes.index(next_node)+2)]}' untuk menguji potensi terpendam.")
                else:
                    print(f"   Reason: Prasyarat untuk '{next_node}' sudah tuntas. Silakan lanjutkan belajar.")
            else:
                print("\n🎉 AI Rekomendasi: Anda telah menguasai semua materi! Luar biasa!")
                
        elif pilihan == '3':
            # SOCRATIC FILTER / ANTI PLAGIARISME
            print(f"\n=======================================================")
            print("💬 AI TUTOR CHAT - SOCRATIC CODE GUARD ACTIVATED")
            print("=======================================================")
            print("Chatbot dilarang memberikan kode solusi mentah (anti-plagiarisme).")
            print("Tanya apa saja (misal: 'tuliskan kode untuk kuis assembly' atau 'bagaimana konsep pointer?')")
            
            user_msg = input("\nAnda: ")
            
            # Cek indikasi minta jawaban kode langsung
            plagiarism_keywords = ["buatkan kode", "tuliskan program", "tulis kode", "minta jawaban", "tuliskan fungsi", "write code", "give solution"]
            if any(k in user_msg.lower() for k in plagiarism_keywords):
                print(f"\n🤖 AI Tutor (INTERVENSI ETIKA #4 - SOCRATIC GUARD):")
                print("❌ Maaf, saya tidak dapat memberikan kode solusi praktikum secara langsung demi menjaga integritas akademik.")
                print("💡 Namun, saya akan membimbing Anda langkah demi langkah menggunakan Metode Socratic:")
                print("   1. Apa tujuan utama dari program yang ingin Anda buat?")
                print("   2. Konsep dasar apa yang sudah Anda ketahui (misal: loop, variabel)?")
                print("   3. Coba tuliskan pseudocode logikanya terlebih dahulu, dan saya akan mengevaluasinya bersama Anda.")
            else:
                print(f"\n🤖 AI Tutor:")
                print(f"   Pertanyaan yang bagus tentang '{user_msg}'. Mari kita diskusikan konsepnya...")
                print("   [AI menjelaskan konsep secara teoritis tanpa memberikan code template gratis]")
                
        elif pilihan == '4':
            # UPDATE BKT & SPACED REPETITION
            print("\nPilih materi yang ingin Anda laporkan nilainya:")
            uncompleted = [n for n in nodes if mastery_scores[n] < 0.8][:5]
            if not uncompleted:
                print("Semua materi sudah selesai.")
                continue
                
            for i, uc in enumerate(uncompleted, 1):
                print(f"  {i}. {uc}")
                
            idx = input("Pilih nomor materi: ")
            if idx.isdigit() and 1 <= int(idx) <= len(uncompleted):
                target = uncompleted[int(idx)-1]
                score_str = input(f"Masukkan nilai ujian/kuis untuk {target} (0-100): ")
                if score_str.isdigit():
                    score_val = float(score_str) / 100.0
                    mastery_scores[target] = score_val
                    print(f"\n✅ Data kognitif ter-update. Mastery {target} = {score_val:.0%}")
                    
                    if score_val < 0.8:
                        remedial_attempts += 1
                        print(f"⚠️ Nilai di bawah KKM (80%). Remedial attempts: {remedial_attempts}")
                    else:
                        remedial_attempts = 0
                        print(f"🎉 Selamat! Anda menguasai {target}.")
                        sr_manager.review_concept(target, 3)
                        
        elif pilihan == '5':
            # REPORT & HUMAN OVERRIDE (LOSS OF TRUST MITIGATION)
            print(f"\n=======================================================")
            print("🛡️ REPORT & HUMAN-IN-THE-LOOP OVERRIDE")
            print("=======================================================")
            print("Jika Anda merasa rekomendasi AI salah atau chatbot berhalusinasi,")
            print("Asisten Praktikum (Manusia) memiliki wewenang penuh untuk meng-override sistem.")
            
            # Calculate current recommendation node
            current_rec = None
            for n in nodes:
                if mastery_scores[n] < 0.8:
                    current_rec = n
                    break
            if not current_rec:
                current_rec = nodes[-1]
                
            print(f"\nPosisi belajar AI saat ini: Rekomendasi = {current_rec}")
            override = input("Apakah Anda ingin mengajukan keberatan / override rekomendasi? (y/n): ")
            if override.lower() == 'y':
                print("\n[VERIFICATION LOOP & ACCOUNTABILITY]")
                reason = input("Masukkan alasan (misal: 'AI merekomendasikan topik yang terlalu susah' atau 'Hallucination'): ")
                print(f"📝 Laporan terkirim ke database Asisten Praktikum: '{reason}'")
                print("\n--- PILIHAN MANUAL OVERRIDE (ASISTEN PRAKTIKUM) ---")
                for i, node in enumerate(nodes[:10], 1):
                    print(f"  {i}. {node}")
                new_idx = input("Pilih materi alternatif yang disetujui Asisten: ")
                if new_idx.isdigit() and 1 <= int(new_idx) <= 10:
                    override_node = nodes[int(new_idx)-1]
                    print(f"\n🚀 OVERRIDE DISAPPROVED SUCCESSFUL!")
                    print(f"   Jalur belajar digeser secara manual ke: {override_node}")
                    print("   [LOSS OF TRUST #5]: Kepercayaan terjaga melalui wewenang penuh manusia atas keputusan AI.")
                    
        elif pilihan == '6':
            # DATA PRIVACY LOG (SECURITY THREATS)
            print(f"\n=======================================================")
            print("🔒 PRIVACY AUDIT & DATA MINIMIZATION LOG")
            print("=======================================================")
            print("Sistem mematuhi aturan privasi data (GDPR / UU PDP):")
            print("1. Identitas siswa disandarkan pada ID acak.")
            print("2. Tidak mencatat data sensitif seperti ras, agama, orientasi seksual, atau status sosial.")
            print("3. Data yang tersimpan di server database:")
            import hashlib
            hashed_id = hashlib.sha256(str(active_student['id']).encode()).hexdigest()[:15]
            print(f"   - Student ID: SHA256({active_student['id']}) -> {hashed_id}...")
            print(f"   - Mastery Vector: {[round(s, 2) for s in mastery_scores.values()][:5]}... (Hanya representasi numerik kognitif)")
            print("   - Activity Clicks: Disimpan teragregasi bulanan (bukan log aktivitas realtime detik demi detik).")
            print("\n✅ Status Keamanan: AMAN & MINIMALIS (Zero Personal Data Leaked).")
            
        elif pilihan == '7':
            print("\nSampai jumpa! Menutup simulasi dengan aman. 🛡️👋")
            break
        else:
            print("\nPilihan tidak valid. Silakan pilih menu 1-7.")
            
start_chatbot_sim(engine)
